# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 03: Data Transformation with Dynamic Tables (Declarative DAG)

---

### What You'll Learn
- Building a transformation DAG using Dynamic Tables (declarative approach)
- Setting TARGET_LAG for different freshness requirements
- Monitoring DT refresh pipelines
- Comparing declarative (DTs) vs imperative (SPs + Tasks) approaches

### Dynamic Tables vs Traditional ETL

| Aspect | Dynamic Tables | Traditional SP + Tasks |
|--------|---------------|----------------------|
| **Definition** | Declarative SQL (just write the query) | Imperative (write INSERT/MERGE logic) |
| **Orchestration** | Automatic (Snowflake manages the DAG) | Manual (define Task dependencies) |
| **Incremental** | Built-in (automatic change tracking) | Must code yourself (MERGE, CDC) |
| **Schema** | Auto-managed | Manual DDL management |
| **Monitoring** | Native DT monitoring views | Task history + custom logging |

### DAG Architecture
```
CUSTOMERS ─────┐
               ├──► CUSTOMER_360 ──────┐
ACCOUNTS ──────┘                       ├──► EXECUTIVE_DASHBOARD
                                       │
TRANSACTIONS ──► TXN_SUMMARY ──────────┘
```

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Dynamic Table - CUSTOMER_360

A unified customer view joining customer profiles with their account summaries. This is the foundation for downstream analytics.

**TARGET_LAG = '1 hour'** - Customer data doesn't change frequently, hourly freshness is sufficient.

In [ ]:
-- =============================================================================
-- DYNAMIC TABLE: CUSTOMER_360
-- Unified customer view with account summaries
-- =============================================================================

CREATE OR REPLACE DYNAMIC TABLE CUSTOMER_360
    TARGET_LAG = '1 hour'
    WAREHOUSE = HOL_MEDIUM_WH
AS
SELECT
    c.CUSTOMER_ID,
    c.FIRST_NAME,
    c.LAST_NAME,
    c.EMAIL,
    c.SEGMENT,
    c.RISK_RATING,
    c.CREATED_AT AS CUSTOMER_SINCE,
    c.IS_ACTIVE,
    -- Account aggregations
    COUNT(DISTINCT a.ACCOUNT_ID) AS TOTAL_ACCOUNTS,
    SUM(CASE WHEN a.STATUS = 'ACTIVE' THEN 1 ELSE 0 END) AS ACTIVE_ACCOUNTS,
    SUM(a.BALANCE) AS TOTAL_BALANCE,
    AVG(a.BALANCE) AS AVG_ACCOUNT_BALANCE,
    MAX(a.LAST_ACTIVITY_DATE) AS LAST_ACTIVITY,
    -- Account type breakdown
    SUM(CASE WHEN a.ACCOUNT_TYPE = 'CHECKING' THEN a.BALANCE ELSE 0 END) AS CHECKING_BALANCE,
    SUM(CASE WHEN a.ACCOUNT_TYPE = 'SAVINGS' THEN a.BALANCE ELSE 0 END) AS SAVINGS_BALANCE,
    SUM(CASE WHEN a.ACCOUNT_TYPE = 'MONEY_MARKET' THEN a.BALANCE ELSE 0 END) AS MONEY_MARKET_BALANCE,
    -- Semi-structured data extraction
    c.ADDRESS_JSON:city::VARCHAR AS CITY,
    c.ADDRESS_JSON:state::VARCHAR AS STATE
FROM CUSTOMERS c
LEFT JOIN ACCOUNTS a ON c.CUSTOMER_ID = a.CUSTOMER_ID
GROUP BY ALL;

## Step 2: Dynamic Table - TXN_SUMMARY

Daily and monthly transaction aggregations by account, category, and channel. Feeds the executive dashboard.

**TARGET_LAG = '30 minutes'** - Transactions are more time-sensitive for fraud detection and real-time dashboards.

In [ ]:
-- =============================================================================
-- DYNAMIC TABLE: TXN_SUMMARY
-- Daily transaction aggregations by account, category, and channel
-- =============================================================================

CREATE OR REPLACE DYNAMIC TABLE TXN_SUMMARY
    TARGET_LAG = '30 minutes'
    WAREHOUSE = HOL_MEDIUM_WH
AS
SELECT
    ACCOUNT_ID,
    DATE_TRUNC('DAY', TO_TIMESTAMP(TXN_DATE)) AS TXN_DAY,
    DATE_TRUNC('MONTH', TO_TIMESTAMP(TXN_DATE)) AS TXN_MONTH,
    CATEGORY,
    CHANNEL,
    TXN_TYPE,
    -- Aggregations
    COUNT(*) AS TXN_COUNT,
    SUM(AMOUNT) AS TOTAL_AMOUNT,
    AVG(AMOUNT) AS AVG_AMOUNT,
    MAX(AMOUNT) AS MAX_AMOUNT,
    MIN(AMOUNT) AS MIN_AMOUNT,
    -- Status breakdown
    SUM(CASE WHEN STATUS = 'COMPLETED' THEN 1 ELSE 0 END) AS COMPLETED_COUNT,
    SUM(CASE WHEN STATUS = 'FAILED' THEN 1 ELSE 0 END) AS FAILED_COUNT,
    SUM(CASE WHEN STATUS = 'REVERSED' THEN 1 ELSE 0 END) AS REVERSED_COUNT,
    -- Derived metrics
    SUM(CASE WHEN AMOUNT > 1000 THEN 1 ELSE 0 END) AS HIGH_VALUE_TXN_COUNT
FROM TRANSACTIONS
GROUP BY ALL;

## Step 3: Dynamic Table - EXECUTIVE_DASHBOARD (Downstream - depends on CUSTOMER_360 and TXN_SUMMARY)

Top-level KPIs combining customer and transaction data. Uses `TARGET_LAG = DOWNSTREAM` to refresh only when upstream DTs have new data.

In [ ]:
-- =============================================================================
-- DYNAMIC TABLE: EXECUTIVE_DASHBOARD
-- Top-level KPIs combining customer 360 and transaction summary
-- Uses DOWNSTREAM lag - refreshes when upstream DTs refresh
-- =============================================================================

CREATE OR REPLACE DYNAMIC TABLE EXECUTIVE_DASHBOARD
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = HOL_MEDIUM_WH
AS
SELECT
    c360.STATE,
    c360.SEGMENT,
    -- Customer KPIs
    COUNT(DISTINCT c360.CUSTOMER_ID) AS TOTAL_CUSTOMERS,
    SUM(CASE WHEN c360.IS_ACTIVE THEN 1 ELSE 0 END) AS ACTIVE_CUSTOMERS,
    AVG(c360.TOTAL_BALANCE) AS AVG_CUSTOMER_BALANCE,
    SUM(c360.TOTAL_BALANCE) AS TOTAL_DEPOSITS,
    -- Transaction KPIs (monthly)
    SUM(ts.TXN_COUNT) AS MONTHLY_TXN_COUNT,
    SUM(ts.TOTAL_AMOUNT) AS MONTHLY_TXN_VOLUME,
    AVG(ts.AVG_AMOUNT) AS AVG_TXN_SIZE,
    SUM(ts.FAILED_COUNT) AS MONTHLY_FAILED_TXNS,
    -- Derived metrics
    ROUND(SUM(ts.FAILED_COUNT) / NULLIF(SUM(ts.TXN_COUNT), 0) * 100, 3) AS FAILURE_RATE_PCT,
    SUM(ts.HIGH_VALUE_TXN_COUNT) AS HIGH_VALUE_TXNS
FROM CUSTOMER_360 c360
LEFT JOIN TXN_SUMMARY ts ON c360.CUSTOMER_ID = (
    SELECT DISTINCT a.CUSTOMER_ID 
    FROM ACCOUNTS a 
    WHERE a.ACCOUNT_ID = ts.ACCOUNT_ID 
    LIMIT 1
)
WHERE ts.TXN_MONTH = DATE_TRUNC('MONTH', CURRENT_DATE())
GROUP BY ALL;

## Step 4: Monitor the Dynamic Table DAG

Snowflake provides built-in monitoring for Dynamic Table refresh pipelines.

In [ ]:
-- =============================================================================
-- MONITOR DYNAMIC TABLE DAG
-- =============================================================================

-- View all Dynamic Tables and their configuration
SHOW DYNAMIC TABLES IN SCHEMA HOL_LAB;

-- Check refresh history
SELECT *
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME => 'CUSTOMER_360'
))
ORDER BY REFRESH_START_TIME DESC
LIMIT 10;

In [ ]:
-- View the DAG dependencies (which DTs feed which)
SELECT *
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_GRAPH_HISTORY())
WHERE QUALIFIED_NAME LIKE '%HOL_LAB%'
ORDER BY VALID_FROM DESC
LIMIT 20;

In [ ]:
-- Manually trigger a refresh to see results immediately
ALTER DYNAMIC TABLE CUSTOMER_360 REFRESH;
ALTER DYNAMIC TABLE TXN_SUMMARY REFRESH;

-- Wait briefly then refresh downstream
-- (In production, downstream DTs refresh automatically)
ALTER DYNAMIC TABLE EXECUTIVE_DASHBOARD REFRESH;

In [ ]:
-- Validate results - Customer 360
SELECT SEGMENT, COUNT(*) AS CUSTOMERS, 
    ROUND(AVG(TOTAL_BALANCE), 2) AS AVG_TOTAL_BALANCE,
    ROUND(AVG(TOTAL_ACCOUNTS), 1) AS AVG_ACCOUNTS_PER_CUSTOMER
FROM CUSTOMER_360
GROUP BY SEGMENT
ORDER BY AVG_TOTAL_BALANCE DESC;

## Summary

### Dynamic Table DAG Created

| Dynamic Table | Source Tables | TARGET_LAG | Purpose |
|--------------|--------------|------------|---------|
| CUSTOMER_360 | CUSTOMERS, ACCOUNTS | 1 hour | Unified customer view |
| TXN_SUMMARY | TRANSACTIONS | 30 minutes | Transaction aggregations |
| EXECUTIVE_DASHBOARD | CUSTOMER_360, TXN_SUMMARY | DOWNSTREAM | Top-level KPIs |

### Key Takeaways
- **No orchestration code needed** - Snowflake automatically determines the refresh order
- **Incremental by default** - only processes changed data
- **DOWNSTREAM lag** - downstream tables only refresh when upstream has new data (cost efficient)
- **Built-in monitoring** - use DYNAMIC_TABLE_REFRESH_HISTORY and DYNAMIC_TABLE_GRAPH_HISTORY

---

**Next:** See `03b_Stored_Procedures_Tasks.ipynb` for the same logic implemented as stored procedures + tasks (imperative approach), then compare.